# PGlite: Postgres without a server

[PGlite](https://pglite.dev/) is Postgres compiled to WebAssembly: the whole database engine runs
inside a JavaScript process, single-user, with no daemon, no port, no Docker. It was built for the
browser and for Node, and [py-pglite](https://pypi.org/project/py-pglite/) wraps it for Python by
starting a Node process that speaks the Postgres wire protocol over a socket, so psycopg and
SQLAlchemy connect to it like to any other Postgres.

What it is good for: unit tests and CI without a database service, demos, notebooks like this one,
and trying an extension (pgvector is bundled) in a few seconds. What it is not: a server. This
notebook shows both halves with the project's own models.

**Prerequisites**: Node.js on the PATH (py-pglite installs the `@electric-sql/pglite` package into
its own directory on first start), and this project's `.venv` as kernel. No container is needed.

In [ ]:
import os
import pathlib
import shutil
import sys
import time

ROOT = next(
    p
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "src" / "db.py").exists()
)
sys.path.insert(0, str(ROOT / "src"))

# py-pglite runs Postgres in Node: it needs `npm` and `node` on the kernel's PATH. A kernel started
# from a VS Code opened via the Dock inherits a bare PATH, so look in the usual install places.
if shutil.which("npm") is None:
    home = pathlib.Path.home()
    candidates = [
        "/opt/homebrew/bin",
        "/usr/local/bin",
        *sorted((home / ".nvm/versions/node").glob("*/bin"), reverse=True),
        home / ".volta/bin",
        home / ".local/share/fnm/aliases/default/bin",
    ]
    os.environ["PATH"] = os.pathsep.join(
        [str(c) for c in candidates if (pathlib.Path(c) / "npm").exists()]
        + [os.environ.get("PATH", "")]
    )
if shutil.which("npm") is None:
    raise RuntimeError(
        "PGlite needs Node.js: install it (brew install node) or rebuild the dev container, then restart the kernel"
    )
print("node", shutil.which("node"))

import pandas as pd
from py_pglite import PGliteConfig
from py_pglite.sqlalchemy import SQLAlchemyPGliteManager
from sqlalchemy import select, text
from sqlalchemy.orm import Session

from models import (
    Base,
    Document,
    Event,
)  # the project's real models: Vector(1536) + JSONB

t0 = time.perf_counter()
mgr = SQLAlchemyPGliteManager(
    PGliteConfig(
        extensions=["pgvector"],
        log_level="WARNING",  # Unix socket transport (default); see section 3 for why not use_tcp
    )
)
mgr.start()
engine = mgr.get_engine()
print(f"PGlite up in {time.perf_counter() - t0:.1f}s at {mgr.get_connection_string()}")

with engine.connect() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
    conn.commit()
    print(conn.execute(text("SELECT version()")).scalar())
    print(
        "pgvector",
        conn.execute(
            text("SELECT extversion FROM pg_extension WHERE extname = 'vector'")
        ).scalar(),
    )

## 1. The project's models, unchanged

`Base.metadata.create_all` against the PGlite engine creates `documents` (with the `Vector(1536)`
column) and `events` (JSONB). This is the whole point for tests: the code under test does not
know it is not talking to a real server.

In [ ]:
t0 = time.perf_counter()
Base.metadata.create_all(engine)
print(f"create_all in {time.perf_counter() - t0:.2f}s:", list(Base.metadata.tables))

with Session(engine) as session:
    session.add_all(
        [
            Event(
                kind="login",
                payload={
                    "user": {"id": 1, "name": "Ada", "country": "NG"},
                    "tags": ["returning"],
                },
            ),
            Event(
                kind="purchase",
                payload={
                    "user": {"id": 1, "name": "Ada", "country": "NG"},
                    "amount": 4500.5,
                    "items": [{"sku": "A-1", "qty": 2}],
                    "tags": ["promo"],
                },
            ),
            Event(kind="error", payload={"code": 500, "tags": ["api"]}),
        ]
    )
    session.commit()
    df = pd.read_sql(
        select(
            Event.id,
            Event.kind,
            Event.payload["user"]["country"].astext.label("country"),
        ).where(Event.payload.contains({"tags": ["promo"]})),
        session.connection(),
    )
df

## 2. pgvector inside WebAssembly

Distances, kNN and even an HNSW index work. The vectors here are generated *on the server*, and
the query vector is taken from the table ("documents similar to doc 1") rather than sent from
Python, for a reason explained in section 3.

In [ ]:
with engine.connect() as conn:
    t0 = time.perf_counter()
    conn.execute(
        text("""
        INSERT INTO documents (id, content, embedding)
        SELECT gen_random_uuid(), 'doc ' || g, v.e::vector
        FROM generate_series(1, 300) AS g
        -- LATERAL + a reference to g, or the subquery runs once and every row gets the same vector
        CROSS JOIN LATERAL (SELECT array_agg(random()::real) AS e FROM generate_series(1, 1536) WHERE g > 0) AS v
    """)
    )
    conn.commit()
    print(
        f"300 x 1536-dim random vectors generated server-side in {time.perf_counter() - t0:.2f}s"
    )

    q = (
        select(Document.embedding).where(Document.content == "doc 1").scalar_subquery()
    )  # stays on the server
    dist = Document.embedding.cosine_distance(q).label("distance")
    t0 = time.perf_counter()
    top = conn.execute(select(Document.content, dist).order_by(dist).limit(3)).all()
    print(
        f"exact kNN over 300 rows: {(time.perf_counter() - t0) * 1000:.1f} ms ->",
        [(c, round(d, 3)) for c, d in top],
    )

    t0 = time.perf_counter()
    conn.execute(
        text("CREATE INDEX ON documents USING hnsw (embedding vector_cosine_ops)")
    )
    conn.commit()
    print(f"HNSW index built in {time.perf_counter() - t0:.2f}s")
    plan = conn.execute(
        text(
            "EXPLAIN SELECT id FROM documents ORDER BY embedding <=> (SELECT embedding FROM documents WHERE content = 'doc 1') LIMIT 3"
        )
    ).all()
    print("\n".join(r[0] for r in plan[:3]))

## 3. Where it stops being Postgres

Four limits matter in practice; the cell below demonstrates the third and then the second, in
that order, because the message-size probe leaves the bridge unusable.

1. **One connection at a time.** PGlite is single-user and py-pglite's bridge accepts exactly one
   client (its own docstring says so). `get_engine()` therefore returns one shared engine with a
   single pooled connection, and everything (sessions, pandas, the test fixture) must go through
   it, one operation after another. No parallelism to measure, no lock behaviour between sessions
   to test. Do not try a second simultaneous connection to see what happens: while writing this,
   it took the whole Python process down with a segmentation fault inside psycopg.
2. **Message size (py-pglite 0.5.3).** The Node bridge hands each socket chunk to PGlite as if it
   were a whole protocol message. Anything larger than one chunk, about 6 KB over the Unix
   socket, breaks the bridge: that connection drops and no new connection succeeds until the
   manager is restarted. A 1536-dimensional vector serialised by psycopg is 12 to 30 KB,
   which is why this notebook generates vectors server-side and never sends one from Python.
   Bulk inserts need many small statements. (`use_tcp=True` raises the limit to about 32 KB but
   proved unstable here, with the same segmentation fault, so the notebook stays on the socket.)
3. **Speed.** WebAssembly on one core, 32-bit. About twice as slow as the native container on a
   simple scan here, more on memory-heavy work, and index builds are the slowest part. Fine for
   tests, meaningless for benchmarks.

4. **One PGlite per machine (py-pglite 0.5.3).** Starting or stopping *any* manager kills every
   other PGlite process on the host, including one owned by a different Python process. Two test
   runs in parallel, or a test run next to this notebook, take each other down. That is why the
   test section comes last in this notebook: it stops this instance.

Also: no extensions beyond the bundled set (pgvector, pg_trgm, uuid-ossp, and a few more), no
replication, no `pg_stat_statements`, and no real filesystem durability guarantees in the
in-memory mode used here.

In [ ]:
import psycopg

# 3. speed: one WebAssembly core
with engine.connect() as conn:
    t0 = time.perf_counter()
    conn.execute(text("SELECT count(*) FROM generate_series(1, 2000000)"))
    print(
        f"count over 2M generated rows: {(time.perf_counter() - t0) * 1000:.0f} ms (native Postgres: ~200 ms on this laptop)"
    )

# 2. message size: the largest parameter the socket transport accepts. Each probe uses its own
#    short-lived psycopg connection (the engine's connection is released first: one client at a time).
#    The first failure breaks the bridge, so this is the last thing this notebook does with `mgr`.
engine.dispose()
largest = None
for kb in (2, 4, 6, 8, 12, 16):
    try:
        with psycopg.connect(mgr.get_dsn(), connect_timeout=5) as c:
            c.execute("SELECT length(%s)", ("x" * kb * 1024,))
        largest = kb
    except psycopg.OperationalError:
        break
print(
    f"largest client message accepted over the Unix socket: {largest} KB (first failure at {kb} KB)"
)
try:
    with psycopg.connect(mgr.get_dsn(), connect_timeout=5) as c:
        c.execute("SELECT 1")
    print("bridge still accepts connections")
except psycopg.OperationalError:
    print(
        "bridge is dead after the oversize message; mgr.restart() would be needed to continue"
    )

## 4. The real use: a test fixture

py-pglite ships pytest fixtures (`pglite_engine`, `pglite_session`, `pglite_sqlalchemy_engine`),
but the pattern is simple enough to own. One database per test session, `create_all` once, and
each test gets a transaction that is rolled back at the end. The file below is written to a
temporary directory and run with pytest from this notebook; copy it into `tests/` to use it.

Running it starts a second PGlite in the pytest process, which (limit 4 above) also stops this
notebook's instance; it was unusable after the message-size probe anyway.

In [ ]:
import subprocess
import tempfile

test_dir = pathlib.Path(tempfile.mkdtemp(prefix="pglite-tests-"))
(test_dir / "conftest.py").write_text(f"""
import sys
sys.path.insert(0, {str(ROOT / "src")!r})

import pytest
from py_pglite import PGliteConfig
from py_pglite.sqlalchemy import SQLAlchemyPGliteManager
from sqlalchemy import text
from sqlalchemy.orm import Session

from models import Base


@pytest.fixture(scope="session")
def engine():
    with SQLAlchemyPGliteManager(PGliteConfig(extensions=["pgvector"], use_tcp=True, tcp_port=54330, log_level="ERROR")) as mgr:
        eng = mgr.get_engine()
        with eng.begin() as conn:
            conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
        Base.metadata.create_all(eng)
        yield eng


@pytest.fixture
def session(engine):
    # every test runs inside a transaction that is rolled back: tests never see each other's rows
    with engine.connect() as conn, conn.begin() as tx:
        yield Session(bind=conn)
        tx.rollback()
""")
(test_dir / "test_events.py").write_text("""
from sqlalchemy import func, select

from models import Event


def test_contains_filter(session):
    session.add(Event(kind="login", payload={"user": {"country": "NG"}}))
    session.add(Event(kind="login", payload={"user": {"country": "GH"}}))
    session.flush()
    n = session.scalar(select(func.count()).where(Event.payload.contains({"user": {"country": "NG"}})))
    assert n == 1


def test_isolation(session):
    # the row from the previous test was rolled back
    assert session.scalar(select(func.count()).select_from(Event)) == 0
""")
t0 = time.perf_counter()
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(test_dir)],
    capture_output=True,
    text=True,
)
print(
    result.stdout.strip().splitlines()[-1],
    f"(wall {time.perf_counter() - t0:.1f}s including PGlite startup)",
)

## 5. When to reach for it

| Need | PGlite (py-pglite) | Docker Postgres |
|---|---|---|
| unit tests, CI without services, a fresh DB per test session | **yes**, starts in about 2 s | slower to start, needs Docker |
| pgvector / JSONB / SQL feature parity for tests | yes, same Postgres 17 core | yes |
| concurrency, locking, PgBouncer, replication, EXPLAIN timings | no | yes |
| large payloads (embeddings sent from Python, bulk loads) | not through py-pglite 0.5.3 (about 6 KB per message) | yes |
| data that must survive | in-memory here; PGlite has a file mode, py-pglite exposes `work_dir` | volumes |

Rule of thumb: PGlite for the test suite and quick experiments, the compose stack for anything
that measures or holds data. `settings.py` keeps the two apart: tests build their engine from the
PGlite manager, the app from `DATABASE_URL`.

In [ ]:
# Cleanup: stops the Node process (already gone after the pytest run, see limit 4) and its temp dir
if mgr.is_running():
    mgr.stop()
print("running =", mgr.is_running())